In [41]:
# Importing libraries

import pandas as pd
import numpy as np
import synapseclient
import os
import tarfile
import gzip

In [ ]:
# Accessing synapse client
syn = synapseclient.Synapse()
syn.login(authToken="")

folder_id = "syn51365303" #id for european ancestory

files_to_download = list(syn.getChildren(folder_id)) # gets all files within folder
print(f"Found {len(files_to_download)} files total")

In [ ]:
# Reading in file of proteins

with open("mr_proteins.txt") as f:
    protein_list = f.read().splitlines()


In [ ]:
## Creating a dictionary of proteins with their strongest pQTL

# Reading in dataset

df = pd.read_csv('FinalSNPsProteins.csv')


In [45]:
df.columns

Index(['rsid', 'source_protein', 'CHROM', 'GENPOS', 'ID', 'ALLELE0', 'ALLELE1',
       'A1FREQ', 'INFO', 'N', 'TEST', 'BETA', 'SE', 'CHISQ', 'LOG10P', 'EXTRA',
       'F_stat', 'chr', 'gene_start', 'gene_end', 'query'],
      dtype='object')

In [46]:
# For each protein, get the SNP with the smallest p-value

lead_pqtls = (df
              .sort_values('LOG10P', ascending= False)
              .groupby('source_protein')
              .first())
lead_pqtls = lead_pqtls.reset_index()


In [47]:
# Getting only the pqtls of sig proteins in MR

pqtls_sig_proteins = lead_pqtls[lead_pqtls['source_protein'].isin(protein_list)]

In [48]:
pqtls_sig_proteins.head()

,source_protein,rsid,CHROM,GENPOS,ID,ALLELE0,ALLELE1,A1FREQ,INFO,N,...,BETA,SE,CHISQ,LOG10P,EXTRA,F_stat,chr,gene_start,gene_end,query
11,ACE,rs6504163,17,63468418,17:61545779:C:T:imp:v1,C,T,0.626506,0.986785,33985,...,-0.359973,0.007822,2118.1500,461.7120,NaN,2118.154012,17,63477061,63498380,17:63468418
12,ACHE,rs314368,7,100850823,7:100448445:T:G:imp:v1,T,G,0.476713,0.987340,33187,...,0.303196,0.007277,1735.8400,378.6510,NaN,1735.845835,7,100889994,100896974,7:100850823
116,APOC1,rs5112,19,44927023,19:45430280:C:G:imp:v1,C,G,0.533320,0.875003,33930,...,0.344868,0.007655,2029.7300,442.5030,NaN,2029.739810,19,44914247,44919349,19:44927023
118,APOE,rs12972156,19,44884202,19:45387459:C:G:imp:v1,C,G,0.149617,0.977423,33995,...,-0.779292,0.010619,5385.8100,1171.4800,NaN,5385.791710,19,44905791,44909393,19:44884202
122,APP,rs8131895,21,26131210,21:27503527:C:A:imp:v1,C,A,0.654117,0.988013,33687,...,-0.072745,0.007910,84.5832,19.4337,NaN,84.583054,21,25880550,26171128,21:26131210


In [ ]:
output_path = "genetic_coloc_snps.csv" # where csv file should be saved

for i, item in enumerate(files_to_download):
  
    # Skips proteins that have already been processed
    protein_name = item['name'].split('_')[0] # extracts the protein from the file name (always the first word)
    if protein_name in protein_list:
        print(f'Processing {i+1}/{len(files_to_download)} : {item["name"]}') # gives a measure of progress
        entity = syn.get(item['id'], downloadLocation="./snp_downloads") # gets the synapse ID and adds it to downloads temporarily
        
        with tarfile.open(entity.path, "r:*") as tar: # opens folder just downloaded
            for member in tar.getmembers():
                if not member.name.endswith('.gz'):
                    continue # Skipping the directories and only accessing the actual data
                with gzip.open(tar.extractfile(member), 'rt') as gz: # Opening each file individually
                    data = pd.read_csv(gz, sep="\s+")  # Reading file
                    protein_df = pqtls_sig_proteins[pqtls_sig_proteins['source_protein'] == protein_name]
                    GENPOS = int(protein_df['GENPOS'].iloc[0])
                    CHROM  = int(protein_df['CHROM'].iloc[0])
                    data['close_to_pqtl'] = ((data['CHROM'] == CHROM) & (data['GENPOS'].between(GENPOS - 1000000, GENPOS + 1000000))) # Gets every snps within 1 Mb of lead pQTL
                    data = data[data['close_to_pqtl']]
                    data = data[data['close_to_pqtl'] == True] # Only keeping those close enough to the pQTL
                    if not data.empty:
                        data['source_protein'] = protein_name  # Adding the name of the protein
                        data.to_csv(output_path, mode='a', header=False, index=False) # Adding to csv
                        protein_had_results = True # changing for this protein to show there were 1+ SNPs
                        
        os.remove(entity.path) # Removing download to save space

print("Protein extraction complete")

Processing 14/2940 : ACE_P12821_OID30696_v1_Inflammation_II.tar


Downloaded syn52361489 to /home/jovyan/Olivia/snp_downloads/ACE_P12821_OID30696_v1_Inflammation_II.tar


Processing 15/2940 : ACHE_P22303_OID30585_v1_Inflammation_II.tar


Downloaded syn52363069 to /home/jovyan/Olivia/snp_downloads/ACHE_P22303_OID30585_v1_Inflammation_II.tar


Processing 152/2940 : APOC1_P02654_OID30749_v1_Inflammation_II.tar


Downloaded syn52361316 to /home/jovyan/Olivia/snp_downloads/APOC1_P02654_OID30749_v1_Inflammation_II.tar


... (35 outputs left out)

Downloaded syn51470474 to /home/jovyan/Olivia/snp_downloads/STX4_Q12846_OID21326_v1_Oncology.tar


Processing 2823/2940 : TXNDC15_Q96J42_OID21495_v1_Oncology.tar


Downloaded syn51468732 to /home/jovyan/Olivia/snp_downloads/TXNDC15_Q96J42_OID21495_v1_Oncology.tar


Protein extraction complete


In [ ]:
# Reading in extracted proteins

snps = pd.read_csv('genetic_coloc_snps.csv', header= None)

In [62]:
# Checking dataset 

snps.head()

,CHROM,GENPOS,ID,ALLELE0,ALLELE1,A1FREQ,INFO,N,TEST,BETA,SE,CHISQ,LOG10P,EXTRA,close_to_pqtl,source_protein
0,17,62468449,17:60545810:T:C:imp:v1,T,C,0.999150,0.605866,33985,ADD,0.144681,0.165768,0.761760,0.417053,NaN,True,ACE
1,17,62468548,17:60545909:T:TA:imp:v1,T,TA,0.925210,0.823782,33985,ADD,-0.035834,0.015745,5.180040,1.641160,NaN,True,ACE
2,17,62468779,17:60546140:C:T:imp:v1,C,T,0.000828,0.744954,33985,ADD,-0.056037,0.151294,0.137187,0.148073,NaN,True,ACE
3,17,62468810,17:60546171:C:G:imp:v1,C,G,0.034182,0.904865,33985,ADD,-0.019916,0.021749,0.838528,0.443917,NaN,True,ACE
4,17,62469069,17:60546430:T:TC:imp:v1,T,TC,0.286424,0.850715,33985,ADD,0.021182,0.009014,5.521930,1.726320,NaN,True,ACE


In [57]:
# adding column names

snps.columns = ['CHROM', 'GENPOS', 'ID', 'ALLELE0', 'ALLELE1', 'A1FREQ',
                'INFO', 'N', 'TEST', 'BETA', 'SE', 'CHISQ', 'LOG10P', 
                'EXTRA', 'close_to_pqtl', 'source_protein']

In [58]:
snps.head()

,CHROM,GENPOS,ID,ALLELE0,ALLELE1,A1FREQ,INFO,N,TEST,BETA,SE,CHISQ,LOG10P,EXTRA,close_to_pqtl,source_protein
0,17,62468449,17:60545810:T:C:imp:v1,T,C,0.999150,0.605866,33985,ADD,0.144681,0.165768,0.761760,0.417053,NaN,True,ACE
1,17,62468548,17:60545909:T:TA:imp:v1,T,TA,0.925210,0.823782,33985,ADD,-0.035834,0.015745,5.180040,1.641160,NaN,True,ACE
2,17,62468779,17:60546140:C:T:imp:v1,C,T,0.000828,0.744954,33985,ADD,-0.056037,0.151294,0.137187,0.148073,NaN,True,ACE
3,17,62468810,17:60546171:C:G:imp:v1,C,G,0.034182,0.904865,33985,ADD,-0.019916,0.021749,0.838528,0.443917,NaN,True,ACE
4,17,62469069,17:60546430:T:TC:imp:v1,T,TC,0.286424,0.850715,33985,ADD,0.021182,0.009014,5.521930,1.726320,NaN,True,ACE


In [ ]:
#Resaving df with headers
snps.to_csv("genetic_coloc_snps.csv", index=False)

In [66]:
# Evaluating number of SNPs and proteins
len(snps)

378015

In [73]:
# number of snps by protein

snps.groupby('source_protein').size()

source_protein
ACE         23880
ACHE        33132
APOC1       36138
APOE        35763
APP         23508
ARHGAP45    27446
BLNK        22576
CBLIF       10743
CD2AP       11284
CTF1         6419
DHRS4L2     11194
DMP1        11006
GNAS        12046
GRN          9373
LILRB2      15667
LRRC37A2     9900
PILRA       10558
PILRB        8651
PRSS8        6167
SERPINB1    12999
SERPINF2    14587
ST13         8593
STX4         6413
TXNDC15      9972
dtype: int64